In [4]:
from base import generator_haar

def init_experiment(n):
    d = 2**n

    # Generate 6^n density matrices
    rho_list = generator_haar.generate_n_qubits_rho_haar(n)
    print(f"Generated {len(rho_list)} of {rho_list[0].shape} rho.")

    # Generate unitary
    unitary = generator_haar.random_unitary(d)
    print(f"Generated {unitary.shape} unitary operators.")
    return rho_list, unitary

In [5]:
import numpy as np
import tensorflow as tf

from base import epsilon_rho
def calculate_rho2_unitary(rho_list, unitary):
    rho2_unitary = []
    for rho in rho_list:
        rho2_unitary.append(epsilon_rho.calculate_from_unitary(rho, unitary))
    return rho2_unitary

def calculate_rho2_dephasing(rho_list, n, gamma):
    rho2 = []
    for rho in rho_list:
        rho2.append(epsilon_rho.calculate_dephasing(rho, n, gamma))
    return rho2

def write_to_file(filename, data):
    """Write TensorFlow tensor data to a text file without truncation."""
    tensor_data = data.numpy() if isinstance(data, tf.Tensor) else data

    # Open the file and write the tensor data
    with open(filename, 'w') as f:
        if isinstance(data, np.ndarray):
            np.savetxt(f, data, fmt="%.6f")
        elif isinstance(data, list):
            for item in data:
                f.write(f"{item}\n")
        else:
            f.write(str(data))
def normalize_density_matrix(rho):
    # Calculate the trace of the density matrix
    trace = np.trace(rho)
    
    # Normalize the density matrix
    if trace != 0:
        normalized_rho = rho / trace
    else:
        raise ValueError("The density matrix has a trace of zero and cannot be normalized.")
    
    return normalized_rho




In [6]:
import os
from base import optimize_algorithm
from base import metrics
experiment_folder = 'results/experiment_new/traceX'

for num_qubits in range(1, 2):
    if (experiment_folder == ''):
        break

    print(f"N={num_qubits}")

    #-----Init experiment-----
    # mean_fide = 0 + 0j
    # while mean_fide.real < 0.85:
    #     rho_list, unitary = init_experiment(num_qubits)
    #     rho2_list = calculate_rho2_dephasing(rho_list, num_qubits, 1)
    #     unitary_res, _ = optimize_algorithm.optimize_adam_unitary_dagger_set(rho_list, rho2_list, unitary, 0.005, num_loop=1000)
    #     rho3_list = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res, rho2_list)
    #     mean_fide = metrics.mean_fidelity(rho3_list, rho_list).numpy()
    #     print(mean_fide)
    # write_to_file(os.path.join(experiment_folder, "rho_list.txt"), rho_list)
    # write_to_file(os.path.join(experiment_folder, "init_unitary.txt"), unitary)
    rho_list = [
    np.array([[0.14364795+2.57800736e-17j, 0.34609293-5.68586052e-02j],
              [0.34609293+5.68586052e-02j, 0.85635205+2.62258235e-17j]]),

    np.array([[0.18453881-2.09650606e-17j, 0.32984386-2.04174601e-01j],
              [0.32984386+2.04174601e-01j, 0.81546119-3.62109073e-17j]]),

    np.array([[ 0.97969528+2.26771768e-17j, -0.11034934+8.78376829e-02j],
              [-0.11034934-8.78376829e-02j,  0.02030472+1.48518504e-18j]]),

    np.array([[ 0.96087281+6.89577879e-17j, -0.02912983-1.91696906e-01j],
              [-0.02912983+1.91696906e-01j,  0.03912719-3.66353739e-18j]]),
   
    np.array([[ 0.03131311+2.38095825e-17j, -0.15769847-7.39174724e-02j],
              [-0.15769847+7.39174724e-02j,  0.96868689-2.09847285e-17j]]),

    np.array([[ 0.05876332+7.52428375e-18j, -0.2180843 +8.80308368e-02j],
              [-0.2180843 -8.80308368e-02j,  0.94123668-3.55421136e-17j]])
    ]

    unitary = np.array([
    [-0.605228+0.000000j, -0.796052+0.000000j],
    [-0.796052-0.000000j,  0.605228+0.000000j]
    ])
    beta = 0.1
    # Create 50 t from 0 to 48
    t_s = np.linspace(100, 0, 50)
    for t in t_s:
        t=round(t, 1)
        
        folder_path_a = os.path.join(experiment_folder, str(num_qubits)+"_qubits_"+"{:06.2f}".format(t)+'_a')
        folder_path_b = os.path.join(experiment_folder, str(num_qubits)+"_qubits_"+"{:06.2f}".format(t)+'_b')
        if not os.path.exists(folder_path_a):
            os.makedirs(folder_path_a)
        if not os.path.exists(folder_path_b):
            os.makedirs(folder_path_b)
        g_a = 1 - np.exp(-2 * beta * t)
        g_b = 1 - np.exp(-2 * beta * t**2)
        print(num_qubits, t)
        rho2_list_a = calculate_rho2_dephasing(rho_list, num_qubits, g_a)
        rho2_list_b = calculate_rho2_dephasing(rho_list, num_qubits, g_b)
        #-----Learn kraus operators-----
        unitary_res_a, costdict = optimize_algorithm.optimize_adam_unitary_dagger_set(rho_list, rho2_list_a, unitary, 0.005, num_loop=500)
        unitary_res_b, _ = optimize_algorithm.optimize_adam_unitary_dagger_set(rho_list, rho2_list_b, unitary, 0.005, num_loop=500)
        
        print(costdict)
        #-----Calculate result data-----
        rho3_list_a = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res_a, rho2_list_a)
        rho3_list_b = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res_b, rho2_list_b)
        mean_fidelity_a = metrics.mean_fidelity(rho3_list_a, rho_list)
        mean_fidelity_b = metrics.mean_fidelity(rho3_list_b, rho_list)

        print(g_a, mean_fidelity_a)
        print(g_b, mean_fidelity_b)
        rho2_a = rho2_list_a[1]
        rho2_out_a = epsilon_rho.calculate_from_unitary(rho2_a, unitary_res_a)
        rho2_b = rho2_list_b[1]
        rho2_out_b = epsilon_rho.calculate_from_unitary(rho2_b, unitary_res_b)
        # Define Pauli-X matrix
        pauli_X = np.array([[0, 1], [1, 0]], dtype=np.complex128)
        # Calculate tr(X1, rho)
        trace_rho2_a = metrics.trace_Pauli(rho2_a, 0, pauli_X)
        trace_out_rho2_a = metrics.trace_Pauli(rho2_out_a, 0, pauli_X)
        trace_rho2_b = metrics.trace_Pauli(rho2_b, 0, pauli_X)
        trace_out_rho2_b = metrics.trace_Pauli(rho2_out_b, 0, pauli_X)

        #-----Write to folder-----    
        write_to_file(os.path.join(folder_path_a,"unitary_res.txt"), unitary_res_a)
        write_to_file(os.path.join(folder_path_b,"unitary_res.txt"), unitary_res_b)

        write_to_file(os.path.join(folder_path_a,"rho2.txt"), rho2_a)
        write_to_file(os.path.join(folder_path_b,"rho2.txt"), rho2_b)
        write_to_file(os.path.join(folder_path_a,"rho2_out.txt"), rho2_out_a)
        write_to_file(os.path.join(folder_path_b,"rho2_out.txt"), rho2_out_b)

        write_to_file(os.path.join(folder_path_a,"trace_rho2.txt"), trace_rho2_a)
        write_to_file(os.path.join(folder_path_b,"trace_rho2.txt"), trace_rho2_b)

        write_to_file(os.path.join(folder_path_a,"trace_rho2_out.txt"), trace_out_rho2_a)
        write_to_file(os.path.join(folder_path_b,"trace_rho2_out.txt"), trace_out_rho2_b)

        write_to_file(os.path.join(folder_path_a,"fidelity.txt"), mean_fidelity_a)
        write_to_file(os.path.join(folder_path_b,"fidelity.txt"), mean_fidelity_b)


    
    

N=1
1 100.0
[np.float64(1.1336577864109405), np.float64(0.9647700671559468), np.float64(0.9546368931390573), np.float64(0.944471533366049), np.float64(0.9342787597008556), np.float64(0.9240616132645956), np.float64(0.9138227265919922), np.float64(0.9035646257452044), np.float64(0.8932898296070576), np.float64(0.8830008900687245), np.float64(0.8727004104182715), np.float64(0.8623910541978357), np.float64(0.8520755492749762), np.float64(0.8417566891995483), np.float64(0.8314373328415181), np.float64(0.8211204028294201), np.float64(0.8108088830827862), np.float64(0.8005058156177478), np.float64(0.7902142967449173), np.float64(0.779937472745833), np.float64(0.7696785350959315), np.float64(0.7594407152915171), np.float64(0.7492272793320923), np.float64(0.7390415219057586), np.float64(0.7288867603230621), np.float64(0.7187663282430835), np.float64(0.7086835692342702), np.float64(0.698641830211395), np.float64(0.6886444547888472), np.float64(0.6786947765892313), np.float64(0.6687961125448774)